# Fraud Detection – Model Building & Evaluation

This notebook builds and evaluates machine learning models to detect
fraudulent transactions in e-commerce and banking datasets.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("../data/processed/fraud_processed.csv")
df.head()

In [ ]:
X = df.drop(columns=['class'])
y = df['class']

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

y_train.value_counts(), y_train_res.value_counts()

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_res, y_train_res)

In [ ]:
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:,1]

In [ ]:
f1_lr = f1_score(y_test, y_pred_lr)
f1_lr

In [ ]:
auc_pr_lr = average_precision_score(y_test, y_prob_lr)
auc_pr_lr

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred_lr),
            annot=True, fmt="d")
plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_res, y_train_res)

In [ ]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

In [ ]:
f1_rf = f1_score(y_test, y_pred_rf)
auc_pr_rf = average_precision_score(y_test, y_prob_rf)

f1_rf, auc_pr_rf

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred_rf),
            annot=True, fmt="d")
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in cv.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
    rf.fit(X_tr_res, y_tr_res)
    preds = rf.predict(X_val)
    f1_scores.append(f1_score(y_val, preds))

np.mean(f1_scores), np.std(f1_scores)

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "F1 Score": [f1_lr, f1_rf],
    "AUC-PR": [auc_pr_lr, auc_pr_rf]
})

results

## Model Selection

Random Forest outperformed Logistic Regression in both F1-score and AUC-PR,
indicating superior ability to capture non-linear fraud patterns.
While Logistic Regression provides interpretability, Random Forest
offers a better balance between performance and business risk reduction.